In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run /Workspace/Users/chrknov6@hotmail.com/formula1/Incremental/00.Configurations

In [0]:
%run "/Workspace/Users/chrknov6@hotmail.com/formula1/Incremental/002.silver helper functions"

In [0]:
bronze_table = f"{catalog}.{bronze_schema}.circuits"
silver_table = f"{catalog}.{silver_schema}.circuits"

In [0]:
from pyspark.sql.functions import col,initcap,current_timestamp,lit

In [0]:
circuits_df = (
                 spark.table(bronze_table)
                      .filter(col("batch_id") == lit(v_batch_id))
                      .drop(col("url"))
                      .withColumnsRenamed(
                           {"circuitid":"circuit_id",
                            "circuitName":"circuit_name",
                            "lat":"latitude",
                            "long":"longitude"}
                      )
                      .dropDuplicates()
                      .filter(col("circuit_id").isNotNull())
                      .withColumn("circuit_name",initcap(col("circuit_name")))
                      .withColumn("locality",initcap(col("locality")))
)

In [0]:
write_to_silver(circuits_df,silver_table,col("t.circuit_id") == col("s.circuit_id"),
                ["circuit_name", "latitude", "longitude", "locality", "country", "filename", "ingestion_time"]
)
              

In [0]:
# from delta.tables import DeltaTable
# delta_table = DeltaTable.forName(spark,silver_table)

# if not spark.catalog.tableExists(silver_table):
#     circuits_df.write.mode("overwrite").saveAsTable(f'{catalog}.{silver_schema}.circuits')
# else:
#     (
#     delta_table.alias("t")
#     .merge(circuits_df.alias("s"),col("s.circuit_id") == col("t.circuit_id"))
#     .whenMatchedUpdate(
#         condition= col("s.batch_id") >= col("t.batch_id"),
#         set = {
#             "circuit_name" : "s.circuit_name",
#             "latitude": "s.latitude",
#             "longitude": "s.longitude",
#             "locality": "s.locality",
#             "country": "s.country",
#             "filename":"s.filename",
#             "ingestion_time":"s.ingestion_time",
#             "updated_timestamp": "s.updated_timestamp"
#         }
#     )
#     .whenNotMatchedInsertAll()
#     .execute()
#     )





In [0]:
spark.table(silver_table).display()